In [0]:
%sql
CREATE EXTERNAL LOCATION olist_ext_loc
URL 'abfss://olistnbound@vrdatabaselearningadls.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL olist_storage_cred);


In [0]:
%sql
GRANT READ FILES
ON EXTERNAL LOCATION olist_ext_loc
TO `account users`;


In [0]:
%sql GRANT WRITE FILES
ON EXTERNAL LOCATION olist_ext_loc
TO `account users`;


In [0]:
%sql
drop volume raw_catalog.ingestion.olist_volume_geolocation

In [0]:
%sql
CREATE EXTERNAL VOLUME raw_catalog.ingestion.olist_volume
LOCATION 'abfss://olistnbound@vrdatabaselearningadls.dfs.core.windows.net/raw_ingestion/';


In [0]:
%sql
CREATE EXTERNAL VOLUME raw_catalog.ingestion.olist_volume_customer
LOCATION 'abfss://olistnbound@vrdatabaselearningadls.dfs.core.windows.net/olist_customers/';

In [0]:
%sql
CREATE EXTERNAL VOLUME raw_catalog.ingestion.olist_volume_geolocation
LOCATION 'abfss://olistnbound@vrdatabaselearningadls.dfs.core.windows.net/olist_geolocation/';

In [0]:
olist_containers = dbutils.fs.ls("abfss://olistnbound@vrdatabaselearningadls.dfs.core.windows.net/")
[file_info for file_info in olist_containers]

In [0]:
%sql
drop volume raw_catalog.ingestion.olist_customers

In [0]:
fina_result =[]
for file_info in olist_containers:
    print(str(file_info.path))
    query = f"CREATE EXTERNAL VOLUME IF NOT EXISTS raw_catalog.ingestion.{str(file_info.name).rstrip("/")} LOCATION '{str(file_info.path)}'"
    result = spark.sql(query).collect()
    fina_result.append((str(file_info.name), result))

print(fina_result)

In [0]:
display(olist_containers)

In [0]:
%sql
GRANT USAGE
ON STORAGE CREDENTIAL olist_storage_cred
TO `account users`;



In [0]:
%sql
-- Create Catalog
create catalog IF NOT EXISTS raw_catalog managed location 'abfss://vrdatabaselearningfilesystem@vrdatabaselearningadls.dfs.core.windows.net/metastore/root' comment 'raw data catalog';

-- Catalog Permissions
GRANT USE CATALOG ON CATALOG raw_catalog TO `account users`;
GRANT CREATE SCHEMA ON CATALOG raw_catalog TO `account users`;

-- Create Schema
CREATE SCHEMA raw_catalog.ingestion;

-- Schema Permissions
GRANT USE SCHEMA ON SCHEMA raw_catalog.ingestion TO `account users`;
GRANT CREATE TABLE ON SCHEMA raw_catalog.ingestion TO `account users`;

-- Storage Credential Access
--GRANT USAGE ON STORAGE CREDENTIAL olist_storage_cred TO `account users`;

-- External Location Access
--GRANT READ FILES ON EXTERNAL LOCATION olist_ext_loc TO `account users`;
--GRANT WRITE FILES ON EXTERNAL LOCATION olist_ext_loc TO `account users`;

In [0]:
%sql
CREATE EXTERNAL LOCATION olist_external_managed_location
URL 'abfss://vrdatabaselearningfilesystem@vrdatabaselearningadls.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL olist_storage_cred);



In [0]:
%sql
GRANT READ FILES
ON EXTERNAL LOCATION olist_external_managed_location
TO `account users`;

In [0]:
%sql GRANT WRITE FILES
ON EXTERNAL LOCATION olist_external_managed_location
TO `account users`;


In [0]:
%sql
DROP CATALOG raw_catalog CASCADE;



In [0]:
%sql
CREATE TABLE IF NOT EXISTS raw_catalog.ingestion.raw_customers (
    customer_id STRING NOT NULL,
    customer_unique_id STRING NOT NULL,
    customer_zip_code_prefix STRING,
    customer_city STRING,
    customer_state STRING
)
USING DELTA;
